In [0]:
from pyspark.sql import functions as F

campaigns_bronze = spark.table(
    "workspace.fashion_bronze.marketing_campaigns_ingested"
)

print("Bronze campaign records:", campaigns_bronze.count())
campaigns_bronze.printSchema()

Bronze campaign records: 200
root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- campaign_objective: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- campaign_budget: double (nullable = true)
 |-- target_segment: string (nullable = true)
 |-- campaign_status: string (nullable = true)
 |-- targeted_customers: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- customers_reached: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- conversions: integer (nullable = true)
 |-- actual_spend: double (nullable = true)
 |-- attributed_revenue: double (nullable = true)
 |-- reach_rate: double (nullable = true)
 |-- click_through_rate: double (nullable = true)
 |-- conversion_rate: double (nullable = true)
 |-- return_on_ad_spend: double (nullable = true)
 |-- 

In [0]:
campaigns_bronze.agg(
    F.count("*").alias("total_campaigns"),
    F.countDistinct("campaign_id").alias("unique_campaigns"),
    F.min("start_date").alias("first_campaign_date"),
    F.max("end_date").alias("last_campaign_date"),
    F.round(F.sum("campaign_budget"), 2).alias("total_budget"),
    F.round(F.sum("actual_spend"), 2).alias("total_actual_spend"),
    F.sum("impressions").alias("total_impressions"),
    F.sum("customers_reached").alias("total_customers_reached"),
    F.sum("clicks").alias("total_clicks"),
    F.sum("conversions").alias("total_conversions"),
    F.round(F.sum("attributed_revenue"), 2).alias("total_attributed_revenue")
).show(truncate=False)

+---------------+----------------+-------------------+------------------+-------------+------------------+-----------------+-----------------------+------------+-----------------+------------------------+
|total_campaigns|unique_campaigns|first_campaign_date|last_campaign_date|total_budget |total_actual_spend|total_impressions|total_customers_reached|total_clicks|total_conversions|total_attributed_revenue|
+---------------+----------------+-------------------+------------------+-------------+------------------+-----------------+-----------------------+------------+-----------------+------------------------+
|200            |200             |2023-01-10         |2026-10-31        |4.644090778E7|3.888342219E7     |13963225         |2766050                |774886      |27180            |1.0855581268E8          |
+---------------+----------------+-------------------+------------------+-------------+------------------+-----------------+-----------------------+------------+-----------------+-

In [0]:
for column in [
    "campaign_type",
    "marketing_channel",
    "campaign_objective",
    "target_segment",
    "campaign_status"
]:
    print(f"\nDistribution: {column}")
    campaigns_bronze.groupBy(column) \
        .count() \
        .orderBy(F.desc("count")) \
        .show(truncate=False)


Distribution: campaign_type
+------------------+-----+
|campaign_type     |count|
+------------------+-----+
|Seasonal          |49   |
|Festive           |45   |
|Promotional       |45   |
|Product Launch    |35   |
|Customer Retention|26   |
+------------------+-----+


Distribution: marketing_channel
+-----------------+-----+
|marketing_channel|count|
+-----------------+-----+
|Instagram Ads    |40   |
|Email            |38   |
|Facebook Ads     |34   |
|Google Ads       |30   |
|WhatsApp         |25   |
|SMS              |16   |
|Organic Social   |10   |
|Organic Search   |7    |
+-----------------+-----+


Distribution: campaign_objective
+--------------------+-----+
|campaign_objective  |count|
+--------------------+-----+
|Sales Conversion    |72   |
|Customer Engagement |40   |
|Brand Awareness     |37   |
|Customer Acquisition|26   |
|Customer Retention  |25   |
+--------------------+-----+


Distribution: target_segment
+--------------------+-----+
|target_segment      |coun

In [0]:
print("Total records:", campaigns_bronze.count())

print("Duplicate campaign IDs:",
      campaigns_bronze
      .groupBy("campaign_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

for column in campaigns_bronze.columns:
    null_count = campaigns_bronze.filter(
        F.col(column).isNull()
    ).count()
    print(f"{column}: {null_count} null values")

Total records: 200
Duplicate campaign IDs: 0
campaign_id: 0 null values
campaign_name: 0 null values
campaign_type: 0 null values
marketing_channel: 0 null values
campaign_objective: 0 null values
start_date: 0 null values
end_date: 0 null values
campaign_budget: 0 null values
target_segment: 0 null values
campaign_status: 0 null values
targeted_customers: 0 null values
impressions: 0 null values
customers_reached: 0 null values
clicks: 0 null values
conversions: 0 null values
actual_spend: 0 null values
attributed_revenue: 0 null values
reach_rate: 0 null values
click_through_rate: 0 null values
conversion_rate: 0 null values
return_on_ad_spend: 0 null values
cost_per_conversion: 0 null values


In [0]:
campaigns_bronze.agg(
    F.sum(F.when(F.col("end_date") < F.col("start_date"), 1).otherwise(0)).alias("invalid_date_ranges"),
    F.sum(F.when(F.col("campaign_budget") <= 0, 1).otherwise(0)).alias("invalid_budgets"),
    F.sum(F.when(F.col("actual_spend") < 0, 1).otherwise(0)).alias("negative_spend"),
    F.sum(F.when(F.col("actual_spend") > F.col("campaign_budget"), 1).otherwise(0)).alias("spend_exceeds_budget"),
    F.sum(F.when(F.col("targeted_customers") <= 0, 1).otherwise(0)).alias("invalid_target_counts"),
    F.sum(F.when(F.col("customers_reached") > F.col("targeted_customers"), 1).otherwise(0)).alias("reach_exceeds_target"),
    F.sum(F.when(F.col("clicks") > F.col("impressions"), 1).otherwise(0)).alias("clicks_exceed_impressions"),
    F.sum(F.when(F.col("conversions") > F.col("clicks"), 1).otherwise(0)).alias("conversions_exceed_clicks"),
    F.sum(F.when(F.col("attributed_revenue") < 0, 1).otherwise(0)).alias("negative_revenue")
).show(truncate=False)

+-------------------+---------------+--------------+--------------------+---------------------+--------------------+-------------------------+-------------------------+----------------+
|invalid_date_ranges|invalid_budgets|negative_spend|spend_exceeds_budget|invalid_target_counts|reach_exceeds_target|clicks_exceed_impressions|conversions_exceed_clicks|negative_revenue|
+-------------------+---------------+--------------+--------------------+---------------------+--------------------+-------------------------+-------------------------+----------------+
|0                  |0              |0             |0                   |0                    |0                   |0                        |0                        |0               |
+-------------------+---------------+--------------+--------------------+---------------------+--------------------+-------------------------+-------------------------+----------------+



In [0]:
campaigns_bronze.agg(
    F.sum(
        F.when(
            F.abs(
                F.col("reach_rate") -
                (F.col("customers_reached") / F.col("targeted_customers") * 100)
            ) > 0.01, 1
        ).otherwise(0)
    ).alias("reach_rate_mismatches"),

    F.sum(
        F.when(
            F.abs(
                F.col("click_through_rate") -
                (F.col("clicks") / F.col("impressions") * 100)
            ) > 0.01, 1
        ).otherwise(0)
    ).alias("ctr_mismatches"),

    F.sum(
        F.when(
            F.abs(
                F.col("conversion_rate") -
                (F.col("conversions") / F.col("clicks") * 100)
            ) > 0.01, 1
        ).otherwise(0)
    ).alias("conversion_rate_mismatches"),

    F.sum(
        F.when(
            F.abs(
                F.col("return_on_ad_spend") -
                (F.col("attributed_revenue") / F.col("actual_spend"))
            ) > 0.01, 1
        ).otherwise(0)
    ).alias("roas_mismatches")
).show(truncate=False)

+---------------------+--------------+--------------------------+---------------+
|reach_rate_mismatches|ctr_mismatches|conversion_rate_mismatches|roas_mismatches|
+---------------------+--------------+--------------------------+---------------+
|0                    |0             |0                         |0              |
+---------------------+--------------+--------------------------+---------------+



In [0]:
campaigns_silver = (
    campaigns_bronze
    .withColumn("campaign_id", F.upper(F.trim(F.col("campaign_id"))))
    .withColumn("campaign_name", F.trim(F.col("campaign_name")))
    .withColumn("campaign_type", F.trim(F.col("campaign_type")))
    .withColumn("marketing_channel", F.trim(F.col("marketing_channel")))
    .withColumn("campaign_objective", F.trim(F.col("campaign_objective")))
    .withColumn("target_segment", F.trim(F.col("target_segment")))
    .withColumn("campaign_status", F.initcap(F.trim(F.col("campaign_status"))))
)

campaigns_silver.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- campaign_objective: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- campaign_budget: double (nullable = true)
 |-- target_segment: string (nullable = true)
 |-- campaign_status: string (nullable = true)
 |-- targeted_customers: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- customers_reached: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- conversions: integer (nullable = true)
 |-- actual_spend: double (nullable = true)
 |-- attributed_revenue: double (nullable = true)
 |-- reach_rate: double (nullable = true)
 |-- click_through_rate: double (nullable = true)
 |-- conversion_rate: double (nullable = true)
 |-- return_on_ad_spend: double (nullable = true)
 |-- cost_per_conversion: double (

In [0]:
print("Total records:", campaigns_silver.count())

print("Duplicate campaign IDs:",
      campaigns_silver
      .groupBy("campaign_id")
      .count()
      .filter(F.col("count") > 1)
      .count())

print("Invalid campaign statuses:",
      campaigns_silver.filter(
          ~F.col("campaign_status").isin("Active", "Completed")
      ).count())

print("Blank campaign names:",
      campaigns_silver.filter(
          F.trim(F.col("campaign_name")) == ""
      ).count())
    

Total records: 200
Duplicate campaign IDs: 0
Invalid campaign statuses: 0
Blank campaign names: 0


In [0]:
campaigns_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_silver.marketing_campaigns")

In [0]:
saved_campaigns = spark.table(
    "workspace.fashion_silver.marketing_campaigns"
)

print("Saved campaign records:", saved_campaigns.count())
saved_campaigns.printSchema()
saved_campaigns.show(5, truncate=False)

Saved campaign records: 200
root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- marketing_channel: string (nullable = true)
 |-- campaign_objective: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- campaign_budget: double (nullable = true)
 |-- target_segment: string (nullable = true)
 |-- campaign_status: string (nullable = true)
 |-- targeted_customers: integer (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- customers_reached: integer (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- conversions: integer (nullable = true)
 |-- actual_spend: double (nullable = true)
 |-- attributed_revenue: double (nullable = true)
 |-- reach_rate: double (nullable = true)
 |-- click_through_rate: double (nullable = true)
 |-- conversion_rate: double (nullable = true)
 |-- return_on_ad_spend: double (nullable = true)
 |-- c

In [0]:
silver_tables = [
    "customers",
    "products",
    "inventory",
    "orders",
    "order_items",
    "marketing_campaigns"
]

for table_name in silver_tables:
    full_table_name = f"workspace.fashion_silver.{table_name}"
    df = spark.table(full_table_name)

    print(
        f"{table_name}: "
        f"{df.count():,} records | "
        f"{len(df.columns)} columns"
    )

customers: 100,000 records | 14 columns
products: 10,000 records | 9 columns
inventory: 40,000 records | 7 columns
orders: 200,000 records | 12 columns
order_items: 499,765 records | 11 columns
marketing_campaigns: 200 records | 22 columns
